# Part 2

In [1]:
# Imports nécessaires
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re
import matplotlib.pyplot as plt
import seaborn as sns


# Configuration de l'affichage
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [2]:
# Création de la session Spark
spark = SparkSession.builder \
    .appName("Apache Log Analysis - Local Mode") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Réduire la verbosité des logs
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.sparkContext.master}")

Spark version: 3.5.0
Spark master: local[*]


In [3]:
hdfs_path = "../data/access.log"  

logs_raw = spark.read.text(hdfs_path)

logs_raw.count()

3602

In [4]:
logs_raw.show(5, truncate=False)


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                 |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|57.70.252.24 - - [01/Nov/2024:00:12:53 +0100] "GET /products/P001 HTTP/1.1" 404 76 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"      |
|168.255.202.227 - - [01/Nov/2024:00:13:40 +0100] "GET /faq HTTP/1.1" 200 19721 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"          |
|109.16.15.24 - - [01/Nov/2024:01:20:27 +0100] "DELETE /contact HTTP/1.1" 2

In [5]:
log_pattern = r'^(\S+) \S+ \S+ \[(.*?)\] "(\S+) (.*?) (HTTP/\S+)" (\d{3}) (\S+) "(.*?)" "(.*?)"'

logs_df = logs_raw.select(
    regexp_extract(col("value"), log_pattern, 1).alias("ip"),
    regexp_extract(col("value"), log_pattern, 2).alias("timestamp"),
    regexp_extract(col("value"), log_pattern, 3).alias("method"),
    regexp_extract(col("value"), log_pattern, 4).alias("url"),
    regexp_extract(col("value"), log_pattern, 6).cast("int").alias("status_code")
).filter(col("ip") != "")

logs_df.cache()

logs_df.printSchema()
logs_df.show(5, truncate=120)

root
 |-- ip: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- status_code: integer (nullable = true)

+---------------+--------------------------+------+--------------+-----------+
|             ip|                 timestamp|method|           url|status_code|
+---------------+--------------------------+------+--------------+-----------+
|   57.70.252.24|01/Nov/2024:00:12:53 +0100|   GET|/products/P001|        404|
|168.255.202.227|01/Nov/2024:00:13:40 +0100|   GET|          /faq|        200|
|   109.16.15.24|01/Nov/2024:01:20:27 +0100|DELETE|      /contact|        200|
| 161.81.216.153|01/Nov/2024:01:35:19 +0100|   GET|    /blog/news|        200|
|  133.231.61.64|01/Nov/2024:01:37:51 +0100|   GET|    /blog/news|        404|
+---------------+--------------------------+------+--------------+-----------+
only showing top 5 rows



In [6]:
logs_df = logs_df \
    .withColumn(
        "status_group",
        when(col("status_code").between(200, 299), "2xx")
        .when(col("status_code").between(400, 499), "4xx")
        .when(col("status_code").between(500, 599), "5xx")
        .otherwise("other")
    ) \
    .withColumn(
        "is_error",
        col("status_code") >= 400
    ) \
    .withColumn(
        "timestamp_parsed",
        to_timestamp(col("timestamp"), "dd/MMM/yyyy:HH:mm:ss Z")
    ) \
    .withColumn(
        "hour",
        hour(col("timestamp_parsed"))
    )

logs_df.select("status_code", "status_group", "is_error", "hour").show(5)


+-----------+------------+--------+----+
|status_code|status_group|is_error|hour|
+-----------+------------+--------+----+
|        404|         4xx|    true|  23|
|        200|         2xx|   false|  23|
|        200|         2xx|   false|   0|
|        200|         2xx|   false|   0|
|        404|         4xx|    true|   0|
+-----------+------------+--------+----+
only showing top 5 rows



profil_ip_df = logs_df.groupBy("ip").agg(
    
    count("*").alias("nb_requetes"),
    
    sum(when(col("is_error") == True, 1).otherwise(0)).alias("nb_erreurs"),
    
    sum(
        when(col("hour").between(1, 4), 1).otherwise(0)
    ).alias("nb_requetes_nuit")
)

profil_ip_df = profil_ip_df.withColumn(
    "taux_erreur",
    round(col("nb_erreurs") / col("nb_requetes"), 3)
)

profil_ip_df = profil_ip_df.select(
    "ip",
    "nb_requetes",
    "nb_erreurs",
    "taux_erreur",
    "nb_requetes_nuit"
)


profil_ip_df.show(5, truncate=False)

In [7]:
df_check = spark.read \
    .format("mongodb") \
    .option("spark.mongodb.connection.uri", "mongodb://mongo-logs:27017") \
    .option("spark.mongodb.database", "logs_db") \
    .option("spark.mongodb.collection", "ip_profiles") \
    .load()


# Part 3

In [10]:
from pymongo import MongoClient
import json

# Connexion au service Mongo e-commerce
client = MongoClient("mongodb://mongo-ecommerce:27017")
db = client["ecommerce_db"]

customers_col = db["customers"]
orders_col = db["orders"]


In [11]:
with open("/home/jovyan/work/data/clients.json", "r", encoding="utf-8") as f:
    clients_data = json.load(f)

# Insérer dans la collection customers
customers_col.insert_many(clients_data)

print(f"{customers_col.count_documents({})} clients insérés")


120 clients insérés


In [12]:
with open("/home/jovyan/work/data/commandes.json", "r", encoding="utf-8") as f:
    commandes_data = json.load(f)

# Insérer dans la collection orders
orders_col.insert_many(commandes_data)

print(f"{orders_col.count_documents({})} commandes insérées")


211 commandes insérées


In [20]:
print("Exemple client :", customers_col.find_one())
print("Exemple commande :", orders_col.find_one())


Exemple client : {'_id': ObjectId('6996da6a6e2e70ab018563ed'), 'client_id': 'C_0012', 'prenom': 'Maryse', 'nom': 'Muller', 'email': 'peltiermichel@example.com', 'ip_address': '23.201.187.4', 'ville': 'Saint Eugènenec', 'date_inscription': '2023-12-18', 'statut': 'actif'}
Exemple commande : {'_id': ObjectId('6996dae268b14c7b63690078'), 'commande_id': 'CMD_00107', 'client_id': 'C_0093', 'date_commande': '2026-02-17T23:03:40.759704', 'statut': 'annulee', 'items': [{'produit_id': 'P001', 'nom_produit': 'Laptop Pro 15', 'quantite': 2, 'prix_unitaire': 1299.99, 'sous_total': 2599.98}, {'produit_id': 'P017', 'nom_produit': 'Routeur WiFi 6', 'quantite': 3, 'prix_unitaire': 179.0, 'sous_total': 537.0}], 'total': 3136.98}


In [22]:
pipeline_total = [
    {"$unwind": "$items"},
    {"$group": {
        "_id": "$client_id",
        "total_depense": {"$sum": {"$multiply": ["$items.quantite", "$items.prix_unitaire"]}}
    }},
    {"$sort": {"total_depense": -1}}
]

total_par_client = list(orders.aggregate(pipeline_total))
for c in total_par_client[:10]:
    print(c)


{'_id': 'C_0041', 'total_depense': 13927.560000000001}
{'_id': 'C_0097', 'total_depense': 12139.14}
{'_id': 'C_0061', 'total_depense': 11358.92}
{'_id': 'C_0024', 'total_depense': 11124.23}
{'_id': 'C_0023', 'total_depense': 10960.050000000001}
{'_id': 'C_0084', 'total_depense': 10484.54}
{'_id': 'C_0007', 'total_depense': 10157.17}
{'_id': 'C_0026', 'total_depense': 8275.55}
{'_id': 'C_0051', 'total_depense': 7873.16}
{'_id': 'C_0087', 'total_depense': 7695.96}


In [25]:
pipeline_panier = [
    {"$unwind": "$items"},
    {"$group": {
        "_id": {"client_id": "$client_id", "order_id": "$_id"},
        "order_total": {"$sum": {"$multiply": ["$items.quantite", "$items.prix_unitaire"]}}
    }},
    {"$group": {
        "_id": "$_id.client_id",
        "panier_moyen": {"$avg": "$order_total"},
        "nb_commandes": {"$sum": 1}
    }},
    {"$sort": {"panier_moyen": -1}}
]

panier_par_client = list(orders.aggregate(pipeline_panier))
for c in panier_par_client[:10]:
    print(c)


{'_id': 'C_0036', 'panier_moyen': 5199.96, 'nb_commandes': 1}
{'_id': 'C_0061', 'panier_moyen': 3786.306666666667, 'nb_commandes': 3}
{'_id': 'C_0097', 'panier_moyen': 3034.785, 'nb_commandes': 4}
{'_id': 'C_0024', 'panier_moyen': 2781.0575, 'nb_commandes': 4}
{'_id': 'C_0013', 'panier_moyen': 2536.49, 'nb_commandes': 1}
{'_id': 'C_0009', 'panier_moyen': 2475.2, 'nb_commandes': 1}
{'_id': 'C_0067', 'panier_moyen': 2355.59, 'nb_commandes': 1}
{'_id': 'C_0066', 'panier_moyen': 2222.29, 'nb_commandes': 2}
{'_id': 'C_0018', 'panier_moyen': 2204.8566666666666, 'nb_commandes': 3}
{'_id': 'C_0084', 'panier_moyen': 2096.9080000000004, 'nb_commandes': 5}


In [28]:
pipeline_top3 = [
    {"$unwind": "$items"},
    {"$group": {
        "_id": "$items.produit_id",
        "total_quantite": {"$sum": "$items.quantite"}
    }},
    {"$sort": {"total_quantite": -1}},
    {"$limit": 3}
]

top3_produits = list(orders.aggregate(pipeline_top3))
for p in top3_produits:
    print(p)


{'_id': 'P009', 'total_quantite': 150}
{'_id': 'P005', 'total_quantite': 106}
{'_id': 'P001', 'total_quantite': 98}


# Part 4